In [1]:
import stlearn as st
import pandas as pd
import pathlib as pathlib
import matplotlib.pyplot as plt

st.settings.set_figure_params(dpi=120)

# Ignore all warnings
import warnings
warnings.filterwarnings("ignore")

/home/xx244/.conda/envs/software/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


/home/xx244/.conda/envs/software/lib/python3.10/site-packages/stlearn/tl/cci/het.py:206: NumbaDeprecationWarning: The keyword argument 'nopython=False' was supplied. From Numba 0.59.0 the default is being changed to True and use of 'nopython=False' will raise a warning as the argument will have no effect. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @jit(parallel=True, nopython=False)


In [2]:
import torch
import anndata as ad
import numpy as np

def get_int_df(adata, use_label, lr=None, sig_interactions=True, title=None):
    """Retrieves the relevant interaction count matrix."""
    no_title = title is None
    labels_ordered = adata.obs[use_label].cat.categories
    if lr is None:  # No LR inputted, so just use all
        int_df = (
            adata.uns[f"lr_cci_{use_label}"]
            if sig_interactions
            else adata.uns[f"lr_cci_raw_{use_label}"]
        )[labels_ordered].loc[labels_ordered]
        title = "Cell-Cell LR Interactions" if no_title else title
    else:
        labels_ordered = adata.obs[use_label].cat.categories
        int_df = (
            adata.uns[f"per_lr_cci_{use_label}"][lr]
            if sig_interactions
            else adata.uns[f"per_lr_cci_raw_{use_label}"][lr]
        )[labels_ordered].loc[labels_ordered]

        title = f"Cell-Cell {lr} interactions" if no_title else title

    return int_df, title

# Mouse brain

In [3]:


df=pd.read_csv("./data/mouse/mouse.csv")
df=df[df['slice_id']=="mouse1_slice201"].copy()
print(df.shape)
genes=torch.load("./data/mouse/genes.pth")
adata=ad.AnnData(X=df[genes].values)
adata.obsm["spatial"]=np.stack([df["centerx"].values,df["centery"].values],axis=-1)
adata.var_names=genes

adata.obs['imagecol']=df["centerx"].values
adata.obs['imagerow']=df["centery"].values
adata.obs["louvain"]=df["subclass"].values
adata.obs["louvain"]=adata.obs["louvain"].astype('category')
adata.uns['spatial']={'dataset':{'use_quality': 'hires', 'scalefactors': {'tissue_hires_scalef': 10/4.71, 'spot_diameter_fullres': 50*10/4.71}}}

(6137, 275)


In [4]:
# QC - Filter genes and cells with at least 10 counts
st.pp.filter_genes(adata, min_counts=10)
st.pp.filter_cells(adata, min_counts=10)

# Store the raw data for using PSTS
adata.raw = adata
# Run PCA, neighbors and clustering.
st.em.run_pca(adata, n_comps=50, random_state=0)
st.pp.neighbors(adata, n_neighbors=25, use_rep='X_pca', random_state=0)
#st.tl.clustering.louvain(adata, random_state=0)

#### Normalize total...
st.pp.normalize_total(adata)

PCA is done! Generated in adata.obsm['X_pca'], adata.uns['pca'] and adata.varm['PCs']


2025-08-02 10:31:49.504577: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-02 10:31:49.841172: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-08-02 10:31:49.841212: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-08-02 10:31:49.841243: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-08-02 10:31:49.905864: I tensorflow/core/platform/cpu_feature_g

2025-08-02 10:31:54.052319: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Created k-Nearest-Neighbor graph in adata.uns['neighbors'] 
Normalization step is finished in adata.X


In [5]:
### Calculating the number of grid spots we will generate
n_ = 125
print(f'{n_} by {n_} has this many spots:\n', n_ * n_)

### Gridding.
grid = st.tl.cci.grid(adata, n_row=n_, n_col=n_, use_label='louvain')
print(grid.shape)  # Slightly less than the above calculation, since we filter out spots with 0 cells.

125 by 125 has this many spots:
 15625
Gridding...


(4787, 254)


In [6]:
# Loading the LR databases available within stlearn (from NATMI)
lrs = st.tl.cci.load_lrs(['connectomeDB2020_lit'], species='mouse')#human!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
print(len(lrs))

# Running the analysis #
st.tl.cci.run(grid, lrs,
              min_spots=20,  # Filter out any LR pairs with no scores for less than min_spots
              distance=None,  # None defaults to spot+immediate neighbours; distance=0 for within-spot mode
              n_pairs=1000,  # Number of random pairs to generate; low as example, recommend ~10,000
              n_cpus=None,   # Number of CPUs for parallel. If None, detects & use all available.
              )

lr_info = grid.uns['lr_summary']  # A dataframe detailing the LR pairs ranked by number of significant spots.
print(lr_info.shape)
print(lr_info)

### Can adjust significance thresholds.
st.tl.cci.adj_pvals(grid, correct_axis='spot',
                    pval_adj_cutoff=0.05, adj_method='fdr_bh')

best_lr = grid.uns['lr_summary'].index.values[0]  # Just choosing one of the top from lr_summary

st.tl.cci.run_cci(grid, 'louvain',  # Spot cell information either in data.obs or data.uns
                  min_spots=2,  # Minimum number of spots for LR to be tested.
                  spot_mixtures=True,  # If True will use the deconvolution data,
                  # so spots can have multiple cell types if score>cell_prop_cutoff
                  cell_prop_cutoff=0.1,  # Spot considered to have cell type if score>0.1
                  sig_spots=True,  # Only consider neighbourhoods of spots which had significant LR scores.
                  n_perms=100,  # Permutations of cell information to get background, recommend ~1000
                  n_cpus=None,
                  )

2293
Calculating neighbours...


0 spots with no neighbours, 587 median spot neighbours.


Spot neighbour indices stored in adata.obsm['spot_neighbours'] & adata.obsm['spot_neigh_bcs'].


Altogether 6 valid L-R pairs


Generating backgrounds & testing each LR pair...:   0%|           [ time left: ? ]

...

Counting celltype-celltype interactions per LR and permuting 100 times.:  67%|██████▋    [ time left: 2:01:30 ]

Counting celltype-celltype interactions per LR and permuting 100 times.:  83%|████████▎  [ time left: 39:33 ]  

Counting celltype-celltype interactions per LR and permuting 100 times.: 100%|██████████ [ time left: 00:00 ]

Counting celltype-celltype interactions per LR and permuting 100 times.: 100%|██████████ [ time left: 00:00 ]

Significant counts of cci_rank interactions for all LR pairs in data.uns[lr_cci_louvain]
Significant counts of cci_rank interactions for each LR pair stored in dictionary data.uns[per_lr_cci_louvain]


In [7]:
int_df, title = get_int_df(grid, "louvain")
print(int_df)

int_df.to_csv("./stLearn/mouse.csv")

            Astro  Endo  L2/3 IT  L4/5 IT  L5 ET  L5 IT  L5/6 NP  L6 CT  \
Astro         582   576     1505     1217      0      0        0    695   
Endo         8190  8645    14558    23772   4041  15707     1162   9593   
L2/3 IT         0     0     9026     5792      0      0        0      0   
L4/5 IT         0  2430     9173    21833   2610   8365        0      0   
L5 ET           0     0        0      838      0      0        0      0   
L5 IT           0     0        0        0    458   1507        0      0   
L5/6 NP         0     0        0      187     42    110       13      0   
L6 CT           0     0        0        0      0      0        0     70   
L6 IT           0     0        0        0      0   1203        0   1809   
L6 IT Car3      0     0        0        0      0      0        0      0   
L6b             0     0        0        0      0      0        0    246   
Lamp5         921  1264     1939     1389    161    683       58      0   
Micro        1171  1704  

# AD

In [8]:
df=pd.read_csv("./data/AD/AD.csv")
df=df[df["section"]=="H20.33.001.CX28.MTG.02.007.1.02.03"].copy()
genes=torch.load("./data/AD/genes.pth")
adata=ad.AnnData(X=df[genes].values)
adata.obs["centerx"]=df["centerx"].values
adata.obs["centery"]=df["centery"].values

adata.var_names=genes
print(adata)

adata.obs['imagecol']=df["centerx"].values
adata.obs['imagerow']=df["centery"].values
adata.obsm["spatial"]=np.stack([df["centerx"].values,df["centery"].values],axis=-1)
adata.obs["louvain"]=df["subclass"].values
adata.obs["louvain"]=adata.obs["louvain"].astype('category')
adata.uns['spatial']={'dataset':{'use_quality': 'hires', 'scalefactors': {'tissue_hires_scalef': 10/4.71, 'spot_diameter_fullres': 50*10/4.71}}}

AnnData object with n_obs × n_vars = 15225 × 140
    obs: 'centerx', 'centery'


In [9]:
# QC - Filter genes and cells with at least 10 counts
st.pp.filter_genes(adata, min_counts=10)
st.pp.filter_cells(adata, min_counts=10)

# Store the raw data for using PSTS
adata.raw = adata
# Run PCA, neighbors and clustering.
st.em.run_pca(adata, n_comps=50, random_state=0)
st.pp.neighbors(adata, n_neighbors=25, use_rep='X_pca', random_state=0)
#st.tl.clustering.louvain(adata, random_state=0)

#### Normalize total...
st.pp.normalize_total(adata)

### Calculating the number of grid spots we will generate
n_ = 125
print(f'{n_} by {n_} has this many spots:\n', n_ * n_)

### Gridding.
grid = st.tl.cci.grid(adata, n_row=n_, n_col=n_, use_label='louvain')
print(grid.shape)  # Slightly less than the above calculation, since we filter out spots with 0 cells.


# Loading the LR databases available within stlearn (from NATMI)
lrs = st.tl.cci.load_lrs(['connectomeDB2020_lit'], species='human')#human!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
print(len(lrs))

# Running the analysis #
st.tl.cci.run(grid, lrs,
              min_spots=20,  # Filter out any LR pairs with no scores for less than min_spots
              distance=None,  # None defaults to spot+immediate neighbours; distance=0 for within-spot mode
              n_pairs=1000,  # Number of random pairs to generate; low as example, recommend ~10,000
              n_cpus=None,   # Number of CPUs for parallel. If None, detects & use all available.
              )

lr_info = grid.uns['lr_summary']  # A dataframe detailing the LR pairs ranked by number of significant spots.
print(lr_info.shape)
print(lr_info)

### Can adjust significance thresholds.
st.tl.cci.adj_pvals(grid, correct_axis='spot',
                    pval_adj_cutoff=0.05, adj_method='fdr_bh')

best_lr = grid.uns['lr_summary'].index.values[0]  # Just choosing one of the top from lr_summary

st.tl.cci.run_cci(grid, 'louvain',  # Spot cell information either in data.obs or data.uns
                  min_spots=2,  # Minimum number of spots for LR to be tested.
                  spot_mixtures=True,  # If True will use the deconvolution data,
                  # so spots can have multiple cell types if score>cell_prop_cutoff
                  cell_prop_cutoff=0.1,  # Spot considered to have cell type if score>0.1
                  sig_spots=True,  # Only consider neighbourhoods of spots which had significant LR scores.
                  n_perms=100,  # Permutations of cell information to get background, recommend ~1000
                  n_cpus=None,
                  )

int_df, title = get_int_df(grid, "louvain")
print(int_df)

int_df.to_csv("./stLearn/AD.csv")

PCA is done! Generated in adata.obsm['X_pca'], adata.uns['pca'] and adata.varm['PCs']


Created k-Nearest-Neighbor graph in adata.uns['neighbors'] 
Normalization step is finished in adata.X
125 by 125 has this many spots:
 15625
Gridding...


(6641, 140)
2293
Calculating neighbours...


0 spots with no neighbours, 223 median spot neighbours.


Spot neighbour indices stored in adata.obsm['spot_neighbours'] & adata.obsm['spot_neigh_bcs'].
Altogether 5 valid L-R pairs


...

Counting celltype-celltype interactions per LR and permuting 100 times.:  60%|██████     [ time left: 1:26:44 ]

Counting celltype-celltype interactions per LR and permuting 100 times.:  80%|████████   [ time left: 30:00 ]  

Counting celltype-celltype interactions per LR and permuting 100 times.: 100%|██████████ [ time left: 00:00 ]

Counting celltype-celltype interactions per LR and permuting 100 times.: 100%|██████████ [ time left: 00:00 ]

Significant counts of cci_rank interactions for all LR pairs in data.uns[lr_cci_louvain]
Significant counts of cci_rank interactions for each LR pair stored in dictionary data.uns[per_lr_cci_louvain]
                 Astrocyte  Chandelier  Endothelial  L2/3 IT  L4 IT  L5 ET  \
Astrocyte                0           0            0        0      0      0   
Chandelier               0           0            0        0      0      0   
Endothelial           1164          50          681     1587    666      0   
L2/3 IT                  0           0          273     2635      0      0   
L4 IT                    0          32          286        0    705      0   
L5 ET                    0           0            0        0      0      7   
L5 IT                    0           0            0        0      0     66   
L5/6 NP                  0           0            0        0      0      0   
L6 CT                  397           0          185        0      0      0   
L6 IT               

# NSCLC

In [10]:
df=pd.read_csv("./data/NSCLC/NSCLC.csv")
df=df[df["section"]=="Lung6"].copy()
print(df.columns)
genes=torch.load("./data/NSCLC/genes.pth")
adata=ad.AnnData(X=df[genes].values)
adata.obs["centerx"]=df['CenterX_global_px'].values
adata.obs["centery"]=df['CenterY_global_px'].values
adata.obsm["spatial"]=np.stack([df['CenterX_global_px'].values,df['CenterY_global_px'].values],axis=-1)
adata.var_names=genes
print(adata)

adata.obs['imagecol']=df['CenterX_global_px'].values
adata.obs['imagerow']=df['CenterY_global_px'].values
adata.obsm["spatial"]=np.stack([df['CenterX_global_px'].values,df['CenterY_global_px'].values],axis=-1)
adata.obs["louvain"]=df["CellType"].values
adata.obs["louvain"]=adata.obs["louvain"].astype('category')
adata.uns['spatial']={'dataset':{'use_quality': 'hires', 'scalefactors': {'tissue_hires_scalef': 8.32/4.71, 'spot_diameter_fullres': 50*8.32/4.71}}}

Index(['Unnamed: 0.1', 'Unnamed: 0', 'fov', 'cell_ID', 'AATK', 'ABL1', 'ABL2',
       'ACE', 'ACE2', 'ACKR1',
       ...
       'SampleID', 'Area', 'AspectRatio', 'CenterX_local_px',
       'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width',
       'Height', 'section'],
      dtype='object', length=976)
AnnData object with n_obs × n_vars = 89948 × 960
    obs: 'centerx', 'centery'
    obsm: 'spatial'


In [11]:
# QC - Filter genes and cells with at least 10 counts
st.pp.filter_genes(adata, min_counts=10)
st.pp.filter_cells(adata, min_counts=10)

# Store the raw data for using PSTS
adata.raw = adata
# Run PCA, neighbors and clustering.
st.em.run_pca(adata, n_comps=50, random_state=0)
st.pp.neighbors(adata, n_neighbors=25, use_rep='X_pca', random_state=0)
#st.tl.clustering.louvain(adata, random_state=0)

#### Normalize total...
st.pp.normalize_total(adata)

### Calculating the number of grid spots we will generate
n_ = 125
print(f'{n_} by {n_} has this many spots:\n', n_ * n_)

### Gridding.
grid = st.tl.cci.grid(adata, n_row=n_, n_col=n_, use_label='louvain')
print(grid.shape)  # Slightly less than the above calculation, since we filter out spots with 0 cells.


# Loading the LR databases available within stlearn (from NATMI)
lrs = st.tl.cci.load_lrs(['connectomeDB2020_lit'], species='human')#human!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
print(len(lrs))

# Running the analysis #
st.tl.cci.run(grid, lrs,
              min_spots=20,  # Filter out any LR pairs with no scores for less than min_spots
              distance=None,  # None defaults to spot+immediate neighbours; distance=0 for within-spot mode
              n_pairs=1000,  # Number of random pairs to generate; low as example, recommend ~10,000
              n_cpus=None,   # Number of CPUs for parallel. If None, detects & use all available.
              )

lr_info = grid.uns['lr_summary']  # A dataframe detailing the LR pairs ranked by number of significant spots.
print(lr_info.shape)
print(lr_info)

### Can adjust significance thresholds.
st.tl.cci.adj_pvals(grid, correct_axis='spot',
                    pval_adj_cutoff=0.05, adj_method='fdr_bh')

best_lr = grid.uns['lr_summary'].index.values[0]  # Just choosing one of the top from lr_summary

st.tl.cci.run_cci(grid, 'louvain',  # Spot cell information either in data.obs or data.uns
                  min_spots=2,  # Minimum number of spots for LR to be tested.
                  spot_mixtures=True,  # If True will use the deconvolution data,
                  # so spots can have multiple cell types if score>cell_prop_cutoff
                  cell_prop_cutoff=0.1,  # Spot considered to have cell type if score>0.1
                  sig_spots=True,  # Only consider neighbourhoods of spots which had significant LR scores.
                  n_perms=100,  # Permutations of cell information to get background, recommend ~1000
                  n_cpus=None,
                  )

int_df, title = get_int_df(grid, "louvain")
print(int_df)

int_df.to_csv("./stLearn/NSCLC.csv")

PCA is done! Generated in adata.obsm['X_pca'], adata.uns['pca'] and adata.varm['PCs']


Created k-Nearest-Neighbor graph in adata.uns['neighbors'] 
Normalization step is finished in adata.X
125 by 125 has this many spots:
 15625
Gridding...


(15385, 960)
2293
Calculating neighbours...


0 spots with no neighbours, 8 median spot neighbours.
Spot neighbour indices stored in adata.obsm['spot_neighbours'] & adata.obsm['spot_neigh_bcs'].


Altogether 517 valid L-R pairs


...

Counting celltype-celltype interactions per LR and permuting 100 times.: 100%|█████████▉ [ time left: 00:01 ]

Counting celltype-celltype interactions per LR and permuting 100 times.: 100%|█████████▉ [ time left: 00:00 ]

Counting celltype-celltype interactions per LR and permuting 100 times.: 100%|██████████ [ time left: 00:00 ]

Counting celltype-celltype interactions per LR and permuting 100 times.: 100%|██████████ [ time left: 00:00 ]

Significant counts of cci_rank interactions for all LR pairs in data.uns[lr_cci_louvain]
Significant counts of cci_rank interactions for each LR pair stored in dictionary data.uns[per_lr_cci_louvain]
              B-cell   NK  T CD4 memory  T CD4 naive  T CD8 memory  \
B-cell           271   55           297          336           163   
NK                35   30            69           62            66   
T CD4 memory     382  117           972          500           418   
T CD4 naive      452   32           628          636           238   
T CD8 memory     141   40           374          180           320   
T CD8 naive      438  101           906          541           344   
Treg             198   58           299          288           191   
endothelial      842  259          1653         1138          1002   
epithelial       675  183          1310          931           930   
fibroblast       766  565          3518         1114           792   
mDC              243   30     

# BC Xenium

In [12]:
df=pd.read_csv("./data/BC/BC.csv")
df=df[df["section"]=="sample1_rep1"].copy()
print(df.columns)
genes=torch.load("./data/BC/genes.pth")

genes=[i for i in genes if i.find("_")<0]

print(genes)
adata=ad.AnnData(X=df[genes].values)
adata.obs["centerx"]=df["centerx"].values
adata.obs["centery"]=df["centery"].values
adata.obsm["spatial"]=np.stack([df["centerx"].values,df["centery"].values],axis=-1)
adata.var_names=genes
print(adata)

adata.obs['imagecol']=df["centerx"].values
adata.obs['imagerow']=df["centery"].values
adata.obsm["spatial"]=np.stack([df["centerx"].values,df["centery"].values],axis=-1)
adata.obs["louvain"]=df["subclass"].values
adata.obs["louvain"]=adata.obs["louvain"].astype('category')
adata.uns['spatial']={'dataset':{'use_quality': 'hires', 'scalefactors': {'tissue_hires_scalef': 1, 'spot_diameter_fullres': 50}}}

Index(['Unnamed: 0', 'ABCC11', 'ACTA2', 'ACTG2', 'ADAM9', 'ADGRE5', 'ADH1B',
       'ADIPOQ', 'AGR3', 'AHSP',
       ...
       'antisense_TRMU', 'antisense_MYLIP', 'antisense_LGI3',
       'antisense_BCL2L15', 'antisense_ADCY4', 'centerx', 'centery',
       'subclass', 'index', 'section'],
      dtype='object', length=327)
['ABCC11', 'ACTA2', 'ACTG2', 'ADAM9', 'ADGRE5', 'ADH1B', 'ADIPOQ', 'AGR3', 'AHSP', 'AIF1', 'AKR1C1', 'AKR1C3', 'ALDH1A3', 'ANGPT2', 'ANKRD28', 'ANKRD29', 'ANKRD30A', 'APOBEC3A', 'APOBEC3B', 'APOC1', 'AQP1', 'AQP3', 'AR', 'AVPR1A', 'BACE2', 'BANK1', 'BASP1', 'BTNL9', 'C15orf48', 'C1QA', 'C1QC', 'C2orf42', 'C5orf46', 'C6orf132', 'CAV1', 'CAVIN2', 'CCDC6', 'CCDC80', 'CCL20', 'CCL5', 'CCL8', 'CCND1', 'CCPG1', 'CCR7', 'CD14', 'CD163', 'CD19', 'CD1C', 'CD247', 'CD27', 'CD274', 'CD3D', 'CD3E', 'CD3G', 'CD4', 'CD68', 'CD69', 'CD79A', 'CD79B', 'CD80', 'CD83', 'CD86', 'CD8A', 'CD8B', 'CD9', 'CD93', 'CDC42EP1', 'CDH1', 'CEACAM6', 'CEACAM8', 'CENPF', 'CLCA2', 'CLDN4', 'CLDN5', 

In [13]:
# QC - Filter genes and cells with at least 10 counts
st.pp.filter_genes(adata, min_counts=10)
st.pp.filter_cells(adata, min_counts=10)

# Store the raw data for using PSTS
adata.raw = adata
# Run PCA, neighbors and clustering.
st.em.run_pca(adata, n_comps=50, random_state=0)
st.pp.neighbors(adata, n_neighbors=25, use_rep='X_pca', random_state=0)
#st.tl.clustering.louvain(adata, random_state=0)

#### Normalize total...
st.pp.normalize_total(adata)

### Calculating the number of grid spots we will generate
n_ = 125
print(f'{n_} by {n_} has this many spots:\n', n_ * n_)

### Gridding.
grid = st.tl.cci.grid(adata, n_row=n_, n_col=n_, use_label='louvain')
print(grid.shape)  # Slightly less than the above calculation, since we filter out spots with 0 cells.


# Loading the LR databases available within stlearn (from NATMI)
lrs = st.tl.cci.load_lrs(['connectomeDB2020_lit'], species='human')#human!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
print(len(lrs))

# Running the analysis #
st.tl.cci.run(grid, lrs,
              min_spots=20,  # Filter out any LR pairs with no scores for less than min_spots
              distance=None,  # None defaults to spot+immediate neighbours; distance=0 for within-spot mode
              n_pairs=1000,  # Number of random pairs to generate; low as example, recommend ~10,000
              n_cpus=None,   # Number of CPUs for parallel. If None, detects & use all available.
              )

lr_info = grid.uns['lr_summary']  # A dataframe detailing the LR pairs ranked by number of significant spots.
print(lr_info.shape)
print(lr_info)

### Can adjust significance thresholds.
st.tl.cci.adj_pvals(grid, correct_axis='spot',
                    pval_adj_cutoff=0.05, adj_method='fdr_bh')

best_lr = grid.uns['lr_summary'].index.values[0]  # Just choosing one of the top from lr_summary

st.tl.cci.run_cci(grid, 'louvain',  # Spot cell information either in data.obs or data.uns
                  min_spots=2,  # Minimum number of spots for LR to be tested.
                  spot_mixtures=True,  # If True will use the deconvolution data,
                  # so spots can have multiple cell types if score>cell_prop_cutoff
                  cell_prop_cutoff=0.1,  # Spot considered to have cell type if score>0.1
                  sig_spots=True,  # Only consider neighbourhoods of spots which had significant LR scores.
                  n_perms=100,  # Permutations of cell information to get background, recommend ~1000
                  n_cpus=None,
                  )

int_df, title = get_int_df(grid, "louvain")
print(int_df)

int_df.to_csv("./stLearn/BC.csv")

PCA is done! Generated in adata.obsm['X_pca'], adata.uns['pca'] and adata.varm['PCs']


Created k-Nearest-Neighbor graph in adata.uns['neighbors'] 
Normalization step is finished in adata.X
125 by 125 has this many spots:
 15625
Gridding...


(14337, 313)
2293
Calculating neighbours...


2 spots with no neighbours, 10 median spot neighbours.


Spot neighbour indices stored in adata.obsm['spot_neighbours'] & adata.obsm['spot_neigh_bcs'].
Altogether 20 valid L-R pairs


...

Counting celltype-celltype interactions per LR and permuting 100 times.:  90%|█████████  [ time left: 00:07 ]

Counting celltype-celltype interactions per LR and permuting 100 times.:  95%|█████████▌ [ time left: 00:02 ]

Counting celltype-celltype interactions per LR and permuting 100 times.: 100%|██████████ [ time left: 00:00 ]

Counting celltype-celltype interactions per LR and permuting 100 times.: 100%|██████████ [ time left: 00:00 ]

Significant counts of cci_rank interactions for all LR pairs in data.uns[lr_cci_louvain]
Significant counts of cci_rank interactions for each LR pair stored in dictionary data.uns[per_lr_cci_louvain]
                         B_Cells  CD4+_T_Cells  CD8+_T_Cells  DCIS_1  DCIS_2  \
B_Cells                     2780          3637          2739      41       0   
CD4+_T_Cells                3485          4534          3800     282       0   
CD8+_T_Cells                2720          3770          2694     150       0   
DCIS_1                         5            33            45     321     365   
DCIS_2                        29             0            44     302     899   
Endothelial                 1116          1450          1324     119      94   
IRF7+_DCs                    185           227           194       0       0   
Invasive_Tumor                 0            46             0       7       2   
LAMP3+_DCs                    81           138           116       0       0   
